# 從零開始做！超簡陋 RAG :)
這邊提供的步驟都是超級省略版，目的導向。  
如果想找完整的是錯過成家超醜但詳細解釋的話，請見 (這裡)[/1004-LLM/exp/aqing]  

為什麼中英夾雜因為老子有時候真的懶得切鍵盤，然後寫英文就是為了裝逼      
為什麼英文的語法像大便因為老子的母語就不是英文  

## Outline
- Set the Environment
- Embedding Model for Single Sentence
    * Tokenizer 
    * Embedding Model (make token embeddings)
    * Pooling
    * (單句) 語意比較
- 打包手作嵌入模型: qingEmbedding()
- 分割大篇文本
- 儲存資料

## Set the Environment
### Hardware
All code in this notebook was tested on MacBook Air without 獨顯.  
The runtime is lightweight (if you use the same LM as I did.) and each cell should finish quickly (usually < 1 min),  
so 電腦的配置似乎並不是非常重要 for this tutorial.
### Internet
The function `transformers.AutoModel.from_pretrained()` will automatically download the required model from Hugging Face.   
Therefore, you need an internet connection when running the code for the first time.  
### Code env
This tutorial is based on **Python**. Required packages are listed below. You can also find detail in `reauirements.txt`.  
It is recommended to install them in a virtual environment like `conda` or something else, using either `pip install` or `conda install`.  
**requirements**
- pathlib
- pytorch (torch)
- time
- transformers

In [ ]:
### --------------------  Import python Models -------------------- ###
from pathlib import Path
import time
import torch
from transformers import AutoTokenizer, AutoModel

timeStart = time.time()

### ----------------------------  Text ---------------------------- ###
root = Path(__file__).resolve().parents[0]
textPath = f'{root}/In-the-Second-Beginning.txt'
textFile = open(textPath, 'r') # Read-only
textFile.close()

testString_1 = "I like astronomy."
testString_2 = "I enjoy watching the night sky full of stars."
testString_3 = "I like tomatoes."
testString_4 = 'I trust the universe will always bring me to you'

## Embedding Model for Single Sentence


### Tokenizer
Tokenizer 中文叫分詞器。可以把自然語言句子先分割成 token, 再轉成 tokenID.  
分割的規則取決于使用的語言模型，英文可能會有拆字跟的情況，比如複數的 's' 自己算一個 token.  
轉成 tokenID 的方法是查字典，字典(voca)也是語言模型自帶的。  

好欸什麼都用別人的，就這個開源爽。

In [ ]:
### --------------------------  Tokenizer ------------------------- ###
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
inp_1 = tokenizer(testString_1, return_tensors='pt')
inp_2 = tokenizer(testString_2, return_tensors='pt')
inp_3 = tokenizer(testString_3, return_tensors='pt') 
inp_4 = tokenizer(testString_4, return_tensors='pt')
re_tokens = tokenizer.convert_ids_to_tokens(inp_4['input_ids'][0]) # tokenID 還原成字, 超樸實的函數命名


#### Tokenizer 的一點解釋
`AutoTokenizer.from_pretrained()` 會去 huggingFace 找到並**下載**語言模型，因為 `transformers` 就是 HF 發行的套件，所以記得聯網。  

這邊使用的語言模型叫做 `all-MiniLM-L6-v2`，是 HF 上語意判斷領域裡面最多人下載的一款模型。  
當然也可以用別的你喜歡的，但就不保證運行的需求和時間ㄌ  

`inp_*` 是將 `teatString_*` 放進分詞器分割的產物，本體是一個類似字典的資料結構。裡面包含 `'input_ids'`（就是 tokenID 本人！） 和 `'attention_mask'`，  
這兩個是有用的等下還要用。

`tokenizer(return_tensors='pt')` 的 `'pt'` 代表 pytorch，接下來的張量運算會用到火炬蟒，所以把 tensor 打包成 pytorch 能認的格式。也可以 `='tf'` for tensorflow.  

### Embedding Model (make token embeddings)
將 tokenIDs 變成 token embeddings (嵌入向量 之 詞向量)  
簡單來說是在嵌入矩陣裡面查表，久遠以前的做法是獨熱編碼和嵌入矩陣相乘，但因為效率的問題所以現在一般是用查表 (indexing) 的。

In [ ]:
### -----------------------  Token Embeddings ----------------------- ###
ebModel = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
torch.set_grad_enabled(False) # 訓練模型(改變)的時候才需要梯度, 只是使用模型的話不用, 所以關掉省電
out_1 = ebModel(**inp_1) # **代表全部帶入, inp 裡面有 'input_ids', 'token_type_ids' ...
out_2 = ebModel(**inp_2)
out_3 = ebModel(**inp_3)
torch.set_grad_enabled(True)

print(out_1.last_hidden_state.shape) # torch.Size([1, 6, 384]) -> (batch size, token num, embedding dim)

#### Embedding Model 的一點解釋
`AutoModel.from_pretrained()` 會去 huggingFace 找到並**下載**語言模型，和 tokenizer 一樣的邏輯。   

把 `inp_*` 丟進嵌入模型 `ebModel` 裡面，回傳值命名為 `out_*`。    

`out_*` 裡面最重要（也是唯一有用的）是 `last_hidden_state`，這神經網路最後一層的輸出，通常來說是語意完整的好東西。
他的形狀可以用 `.shape` or `.size()` 查看，print 出來的順序是 (batch size, token num, embedding dim)  
- batch size: 不知道怎樣但好像都是 1
- token num: 一個句子 (testString_*) 被分割成幾個 token
- embedding dim: 隱藏層的大小(維度, hidden_size, hidden_dim)，每個 token 經過模型後被轉換成 384 維 (this ebmodel) 的向量表示

### Pooling (make sentence embedding)
壓

In [ ]:
def meanPooling(eb_model_output, attention_mask): 
    token_emb = eb_model_output.last_hidden_state
    atMaskP1d = attention_mask.unsqueeze(-1) # 增加一個維度, 和 token_emb 對齊 (attention mask plus 1 dim)
    atMaskP1d = atMaskP1d.expand(token_emb.shape).float() # 最後一個維度沿著 token_emb.shape 複製, 總之對齊造型
                                                          # float() 因為同是浮點數才能運算, numpy 基操
    efficient_token_emb = token_emb * atMaskP1d # 與遮罩相乘, 遮罩=1 的地方才留下值
    poolingResult = torch.sum(efficient_token_emb, dim=1) # 沿著張量相乘結果的的第1條軸(seq_len)加總, 
                                                          # 即所有真實 token 向量的總和, 耶 pooling
    poolingResult_mean = poolingResult / torch.clamp(atMaskP1d.sum(dim=1), min=1e-9) # 平均
                                         # atMaskP1d.sum(dim=1)是沿著注意力遮罩的第1條軸相加, 
                                         # 因為都是 1, 0 所以相加的值就是有效的 token 的數量
                                         # torch.clamp() 是除數!=0保護機制, 等於零的話就當作 1e-9處理
    return poolingResult_mean

sentence_emb_1 = meanPooling(out_1, inp_1['attention_mask']) # Perform pooling, 把 N 個 token_emb 壓成一條 sentence_emb
sentence_emb_1 = torch.nn.functional.normalize(sentence_emb_1, p=2, dim=1) # Normalize embeddings
sentence_emb_2 = meanPooling(out_2, inp_2['attention_mask'])
sentence_emb_3 = meanPooling(out_3, inp_3['attention_mask'])

### Sentense Similarity

In [ ]:
### ------------------------------  Similarity ------------------------------ ###  
cosSim_11 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_1)                                 
cosSim_12 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_2)
cosSim_13 = torch.nn.functional.cosine_similarity(sentence_emb_1, sentence_emb_3)
print('cosine')
print(f'string1 & string1: {cosSim_11[0]}') # 因為就是一樣的句子啊屁眼
print(f'string1 & string2: {cosSim_12[0]:.2f}')
print(f'string1 & string3: {cosSim_13[0]:.2f}') # 成功證明嘻嘻

---

## 打包手作嵌入模型: qingEmbedding()

In [ ]:
def qingsEmbedder(theItem): # theStr: 要轉成embedding的東西
    torch.set_grad_enabled(False)
    inp = tokenizer(theItem, return_tensors="pt")
    out = ebModel(**inp)
    token_emb = out.last_hidden_state
    atMaskP1d = inp['attention_mask'].unsqueeze(-1).expand(token_emb.shape).float()
    torch.set_grad_enabled(True)
    return torch.sum(token_emb * atMaskP1d, dim=1) / torch.clamp(atMaskP1d.sum(dim=1), min=1e-9)

# Demo
cosSim_12 = torch.nn.functional.cosine_similarity(qingsEmbedder(testString_1), qingsEmbedder(testString_2))
cosSim_13 = torch.nn.functional.cosine_similarity(qingsEmbedder(testString_1), qingsEmbedder(testString_3))
print(f'string1 & string2: {cosSim_12[0]:.2f}')
print(f'string1 & string3: {cosSim_13[0]:.2f}')

---

## 分割大篇文本
終於要用到我們的 `textFile`!